## 0 · Vertrekpunt: de lijst is goed, maar waarom?

In het vorige notebook heb je gemeten of jullie recommender een goede lijst geeft. Een cijfer als "hit = 0,7" zegt niet **waarom** een titel in de lijst staat. Een lezer wil dat weten ("waarom krijg ik dit?"), en jij ook: als een titel er ten onrechte in staat, of een goede titel ontbreekt, wil je zien waar het misgaat. Dat heet *explainability*: uitleggen waarom het systeem doet wat het doet.

Jullie recommender is hiervoor geschikt gebouwd. Een profiel en een titel zijn rijen met getallen; de cosine tussen die twee is een som over kolommen, en elke kolom is een **woord of een categorie**. Je kunt dus wijzen op de woorden die een match maken. Dat kan niet meer bij een embeddingmodel, en is een reden om TF-IDF te houden.

Dit notebook gebruikt dezelfde catalogus en dezelfde zeven testlezers als de evaluatie. Om zelfstandig te draaien bevat het hieronder een compacte referentie-recommender (tekst, genre en trefwoorden). Jullie eigen recommender werkt hetzelfde; het slot van dit notebook laat zien hoe je de functies daarin zet.

| Deel | Demo | Opdracht |
|---|---|---|
| 0 | Vertrekpunt: catalogus, testlezers, referentie-recommender | – |
| 1 | Waarom uitleggen, en wat is een goede uitleg? | – |
| 2 | De cosine als som over kolommen | Opdracht 1: `contributions` |
| 3 | Kolomnamen: welk getal hoort bij welk woord? | Opdracht 2: `feature_names` |
| 4 | De woorden achter een aanbeveling | Opdracht 3: `explain` |
| 5 | Welk blok droeg bij: tekst, genre of trefwoorden? | Opdracht 4: `explain_by_block` |
| 6 | Uitleg voor de zeven testlezers | Opdracht 5: een opvallende uitleg zoeken |
| 7 | Van getallen naar een zin voor de lezer | Opdracht 6: `explain_text` |
| 8 | Uitleg als hulpmiddel om missers te verklaren | Reflectie |

In [ ]:
import re
import numpy as np
import pandas as pd
import nltk
from nltk.corpus import stopwords
from scipy.sparse import hstack
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.preprocessing import normalize
from sklearn.metrics.pairwise import cosine_similarity

CATALOG_PATH = "datasets/processed/catalogus_opgeschoond.csv"     # pas aan naar jullie eigen bestand

catalog = pd.read_csv(CATALOG_PATH, sep=";").set_index("cataloguscode")
CODES = catalog.index.tolist()
POS = {code: i for i, code in enumerate(CODES)}                   # cataloguscode -> rijnummer in X
print(len(catalog), "titels in de catalogus")

In [ ]:
def codes(*numbers):
    """Catalogue codes from their numbers: codes(1, 17) -> ["CAT-00001", "CAT-00017"]."""
    return [f"CAT-{n:05d}" for n in numbers]

# De zeven vaste testlezers uit de evaluatie (zonder aanpassingen overgenomen).
daan = {"name": "Daan", "tags": ["thriller", "mysterie", "geheimen"], "genres": ["fictie"],
        "free_text": "een spannende thriller of detective met geheimen en een onverwachte ontknoping",
        "level": "2F", "avoid": [],
        "relevant": codes(54, 57, 66, 87, 89, 94, 98, 101, 102, 103, 108, 109, 110, 114, 121, 123, 124, 125,
                          133, 134, 135, 191, 202, 207, 240, 247, 253, 266, 277)}
thomas = {"name": "Thomas", "tags": ["oorlog", "WOII", "onderduik"], "genres": ["fictie"],
          "free_text": "waargebeurde verhalen uit de Tweede Wereldoorlog, over onderduiken en overleven",
          "level": "3F", "avoid": [], "relevant": codes(1, 10, 11, 17, 32, 33, 42, 46, 85, 248)}
fatima = {"name": "Fatima", "tags": ["migratie", "hoop"], "genres": ["fictie"],
          "free_text": "verhalen over mensen die naar een nieuw land komen en daar opnieuw moeten beginnen",
          "level": "2F", "avoid": [], "relevant": codes(22, 43, 77, 188, 206, 217, 222, 223, 271, 277)}
milan = {"name": "Milan", "tags": ["sport"], "genres": [],
         "free_text": "boeken over sport, en ik wil ook weten wat AI voor mij gaat betekenen",
         "level": "3F", "avoid": [],
         "relevant": codes(48, 106, 111, 115, 184, 240, 268, 272, 139, 140, 148, 152, 153, 175, 178, 183, 209, 280)}
lotte = {"name": "Lotte", "tags": [], "genres": [],
         "free_text": "ik zoek iets over rouw: iemand die een ouder of een vriend verliest en daarmee moet leren leven",
         "level": "3F", "avoid": [], "relevant": codes(21, 79, 117, 136, 198, 201, 205, 213, 219, 234, 276)}
youssef = {"name": "Youssef", "tags": ["criminaliteit", "recht"], "genres": [],
           "free_text": "hoe werkt de rechtspraak, en wat gebeurt er met jongeren die de fout in gaan",
           "level": "3F", "avoid": ["Online artikel", "Blogpost"],
           "relevant": codes(50, 64, 69, 71, 73, 74, 76, 78, 87, 91, 93)}
emma = {"name": "Emma", "tags": [], "genres": ["po\u00ebzie"],
        "free_text": "gedichten over identiteit en opgroeien", "level": "2F", "avoid": [],
        "relevant": codes(160, 161, 252)}

readers = [daan, thomas, fatima, milan, lotte, youssef, emma]
pd.DataFrame({"name": [r["name"] for r in readers], "level": [r["level"] for r in readers],
              "tags": [", ".join(r["tags"]) for r in readers], "genres": [", ".join(r["genres"]) for r in readers]})

### De referentie-recommender

Eén rij per titel in `X`, met drie blokken naast elkaar. Elk blok is eerst genormaliseerd (lengte 1) en daarna met een gewicht vermenigvuldigd. Dat is dezelfde opbouw als jullie eigen `build_vectors`.

| Blok | Bron | Vectorizer | Gewicht |
|---|---|---|---|
| `tekst` | `Korte omschrijving` | TF-IDF, zonder stopwoorden | 1,0 |
| `genre` | `genre` | aanwezig of niet (0/1) | 0,3 |
| `trefwoorden` | `trefwoorden` | aanwezig of niet (0/1), per trefwoord als geheel | 0,5 |

Let op `fillna("")`: een titel zonder genre of trefwoorden heeft in de CSV een lege cel (`NaN`), en een vectorizer accepteert alleen tekst. Een profiel van een lezer is een rij in dezelfde kolommen als `X`, gemaakt met dezelfde vectorizers, zodat kolom *j* voor een lezer en voor een titel hetzelfde woord is.

In [ ]:
nltk.download("stopwords", quiet=True)
CUSTOM_STOPWORDS = {"boek", "verhaal", "verhalen", "roman", "thema", "jongeren", "jonge", "mensen", "wereld", "leven",
                    "nieuwe", "belangrijke", "persoonlijke", "centraal", "komt", "laat", "maakt", "zoals", "gaat"}
STOPWORDS = set(stopwords.words("dutch")) | CUSTOM_STOPWORDS

def tokenize_nostop(text):
    return [t for t in re.findall(r"\w+", str(text).lower()) if t not in STOPWORDS]

def split_keywords(text):
    return [t.strip().lower() for t in str(text).split(";") if t.strip()]

BLOCKS = ["tekst", "genre", "trefwoorden"]                        # de volgorde waarin de blokken in X staan
SOURCE = {"tekst": "Korte omschrijving", "genre": "genre", "trefwoorden": "trefwoorden"}
WEIGHTS = {"tekst": 1.0, "genre": 0.3, "trefwoorden": 0.5}
LEVELS = {"2F": 0, "3F": 1, "3F+": 2}

def build_vectors(catalog, weights=WEIGHTS):
    """Return (X, vectorizers): one sparse row per title, and the fitted vectorizer per block."""
    vecs = {
        "tekst": TfidfVectorizer(tokenizer=tokenize_nostop, token_pattern=None, lowercase=False),
        "genre": CountVectorizer(tokenizer=split_keywords, token_pattern=None, lowercase=False, binary=True),
        "trefwoorden": CountVectorizer(tokenizer=split_keywords, token_pattern=None, lowercase=False, binary=True),
    }
    blocks = [weights[b] * normalize(vecs[b].fit_transform(catalog[SOURCE[b]].fillna("")).astype(float))
              for b in BLOCKS]
    return hstack(blocks).tocsr(), vecs

X, VECS = build_vectors(catalog)

def profile_for(person, weights=WEIGHTS):
    """A reader as one row with the same columns as X (tags, genres and free_text, each in its own block)."""
    text = {"tekst": person.get("free_text", ""),
            "genre": "; ".join(person.get("genres", [])),
            "trefwoorden": "; ".join(person.get("tags", []))}
    blocks = [weights[b] * normalize(VECS[b].transform([text[b]]).astype(float)) for b in BLOCKS]
    return hstack(blocks).tocsr()

def allowed(person):
    """The hard filters: nothing above the reader's level, no type from avoid."""
    level = catalog["Niveau"].map(LEVELS)
    ok = level.isna() | (level <= LEVELS[person["level"]])
    return (ok & ~catalog["Type materiaal"].isin(person.get("avoid", []))).to_numpy()

def recommend(person, n=10):
    """A list of n catalogue codes, best first. Never looks at person['relevant']."""
    sim = cosine_similarity(profile_for(person), X).ravel()
    sim[~allowed(person)] = -1
    return [CODES[j] for j in np.argsort(-sim)[:n]]

def show(code_list):
    return catalog.loc[code_list, ["Titel", "Type materiaal", "Niveau", "genre", "trefwoorden"]]

print("X:", X.shape, "(titels x kolommen)")
show(recommend(daan, n=5))

## 1 · Waarom uitleggen, en wat is een goede uitleg?

Een recommender die alleen een lijst geeft is een *black box*: je ziet wat erin gaat (de lezer) en wat eruit komt (de lijst), niet wat ertussen gebeurt. Een uitleg maakt er een *glass box* van. Drie redenen om dat te willen:

- **Vertrouwen.** "Omdat je *Het dagboek van Anne Frank* koos" is een reden die een lezer begrijpt en kan afkeuren.
- **Fouten vinden.** Staat er een titel in de lijst die er niet hoort, dan zie je aan de uitleg welk woord de schuldige is.
- **Eerlijkheid.** Als het systeem op iets anders let dan jij denkt (een woord dat in elke omschrijving staat), zie je dat alleen als je het kunt aanwijzen.

Een goede uitleg is:

| Eigenschap | Vraag |
|---|---|
| **Waar** | Is dit echt wat het systeem berekende, of een verhaal erbij verzonnen? |
| **Begrijpelijk** | Zou de lezer deze woorden herkennen als reden? |
| **Kort** | Hoeveel redenen kan een lezer in één blik lezen? (meestal 2 à 3) |

In dit notebook is de uitleg **waar** doordat hij uit dezelfde berekening komt als de cosine zelf. Hoe **begrijpelijk** en **kort** hij is, bepaal jij in deel 7.

## 2 · De cosine is een som over kolommen

De cosine tussen een profiel $p$ en een titel $x$ is

$$\cos(p, x) = \frac{p \cdot x}{\lVert p \rVert \, \lVert x \rVert} = \sum_j \frac{p_j \, x_j}{\lVert p \rVert \, \lVert x \rVert}$$

Het inproduct $p \cdot x$ is een som over alle kolommen $j$, en elke term $p_j x_j$ hoort bij één woord of categorie. Alleen kolommen waar **profiel én titel** een waarde hebben, doen mee: wat de lezer niet noemt of de titel niet bevat, is 0 en draagt niets bij. Deel je elke term door $\lVert p \rVert \lVert x \rVert$, dan tellen de termen precies op tot de cosine. Die termen noemen we de **bijdragen**.

Hieronder voor Daan en zijn nummer 1, met de hand uitgerekend.

In [ ]:
p = profile_for(daan)
top1 = recommend(daan, n=1)[0]

p_row = p.toarray().ravel()                          # profiel als gewone rij getallen
x_row = X[POS[top1]].toarray().ravel()               # de titel als gewone rij getallen

terms = p_row * x_row / (np.linalg.norm(p_row) * np.linalg.norm(x_row))   # één term per kolom
print(catalog.loc[top1, "Titel"])
print("kolommen met een term > 0:", int((terms > 0).sum()), "van", len(terms))
print("som van de termen:", round(terms.sum(), 4))
print("cosine_similarity:", round(cosine_similarity(p, X[POS[top1]])[0, 0], 4))

De som en de cosine zijn gelijk, en bijna alle termen zijn 0: van de honderden kolommen delen profiel en titel er maar een handvol. Die handvol is de uitleg.

> **Opdracht 1 · contributions.** Schrijf `contributions(profile, code)`: een rij met per kolom de bijdrage aan de cosine tussen het profiel en de titel met die cataloguscode. De cel na de opdracht is de test: de bijdragen moeten optellen tot de cosine.
>
> **Hier verwacht ik een antwoord op:** `contributions` die de test doorstaat · in je eigen woorden: waarom is de bijdrage 0 voor een woord dat de lezer niet noemt, ook als de titel het woord wel heeft?

In [ ]:
def contributions(profile, code):
    """Per column: how much does it add to the cosine between the profile and the title?"""
    # - Pak de rij van de titel in X: X[POS[code]]
    # - Zet profiel en titelrij om naar gewone rijen: .toarray().ravel()
    # - Vermenigvuldig ze element voor element
    # - Deel door de lengte van beide rijen (np.linalg.norm), anders is de som geen cosine
    # - Is een lengte 0 (een leeg profiel)? Geef dan een rij nullen terug.
    raise NotImplementedError

In [ ]:
# test: de bijdragen tellen op tot de cosine, voor de top 3 van elke testlezer
for person in readers:
    p = profile_for(person)
    for code in recommend(person, n=3):
        c = contributions(p, code)
        assert c.shape == (X.shape[1],), c.shape
        assert (c >= 0).all(), "bijdragen zijn nooit negatief"
        assert abs(c.sum() - cosine_similarity(p, X[POS[code]])[0, 0]) < 1e-9, (person["name"], code)
print("contributions klopt: de bijdragen tellen op tot de cosine")

## 3 · Kolomnamen: welk getal hoort bij welk woord?

`contributions` geeft een rij getallen, maar een uitleg heeft **namen** nodig. Kolom *j* van `X` is een woord uit het tekstblok, een genre of een trefwoord. De vectorizers kennen die namen (`get_feature_names_out()`), per blok.

De valkuil: `X` is de blokken **achter elkaar geplakt**, dus de namen moeten in precies dezelfde volgorde en met precies dezelfde breedte achter elkaar. Staat er één naam te veel of te weinig, of een blok in een andere volgorde, dan hoort elk getal bij het verkeerde woord. Er komt dan geen foutmelding, maar de uitleg is onjuist. Daarom zet je een blokprefix voor elke naam (`tekst:oorlog`, `trefwoorden:woii`): dat laat ook zien uit welk blok een woord komt.

> **Opdracht 2 · feature_names.** Schrijf `feature_names(vecs)`: de lijst met namen van alle kolommen van `X`, in de volgorde van `BLOCKS`, elk met de blokprefix. De test controleert de lengte. Waarom is dat een goede controle?
>
> **Hier verwacht ik een antwoord op:** `feature_names` die de test doorstaat · wat er met de uitleg gebeurt als je de blokken in de verkeerde volgorde zet

In [ ]:
def feature_names(vecs):
    """The names of all columns of X: block prefix + word, in the order the blocks are glued together."""
    # - Loop over BLOCKS, in die volgorde
    # - Haal per blok de namen op met vecs[blok].get_feature_names_out()
    # - Zet de blokprefix ervoor: f"{blok}:{woord}"
    # - Geef een lijst terug
    raise NotImplementedError

In [ ]:
NAMES = feature_names(VECS)
assert len(NAMES) == X.shape[1], (len(NAMES), X.shape[1])
assert len(set(NAMES)) == len(NAMES), "namen moeten uniek zijn (dankzij de blokprefix)"
assert NAMES[0].startswith("tekst:") and NAMES[-1].startswith("trefwoorden:")
print("feature_names klopt:", len(NAMES), "namen voor", X.shape[1], "kolommen")

## 4 · De woorden achter een aanbeveling

Met bijdragen en namen kun je de vraag beantwoorden: welke **k** kolommen dragen het meest bij aan de match? Dat zijn de woorden die de lezer ziet als "omdat ...".

Twee dingen om op te letten:
- Neem alleen kolommen met een bijdrage **groter dan 0**. Heeft een titel maar één woord gemeen met het profiel, dan staan er geen drie redenen; vul de lijst dan niet aan met nullen.
- Sorteer van groot naar klein (`np.argsort(-c)`).

> **Opdracht 3 · explain.** Schrijf `explain(profile, code, k=3)`: een lijst van maximaal `k` paren `(naam, bijdrage)`, de grootste bijdrage eerst, bijdrage afgerond op drie decimalen. Draai hem daarna op de nummer 1 van een testlezer en kijk of de woorden kloppen met wat die lezer vroeg.
>
> **Hier verwacht ik een antwoord op:** `explain` die de test doorstaat · de drie woorden voor de nummer 1 van één testlezer, met je oordeel of dit een goede reden is

In [ ]:
def explain(profile, code, k=3):
    """The k columns (words or categories) that contribute most to the cosine between profile and title."""
    # - Bereken de bijdragen met contributions(profile, code)
    # - Sorteer de kolomnummers van groot naar klein
    # - Houd alleen kolommen over met een bijdrage > 0, en pak de eerste k
    # - Geef per kolom (naam uit NAMES, bijdrage afgerond op 3 decimalen) terug
    raise NotImplementedError

In [ ]:
def occurs(name, code):
    """Does the word or category really occur in the title? (the explanation must be true)"""
    block, word = name.split(":", 1)
    row = catalog.loc[code]
    if block == "tekst":
        return word in tokenize_nostop(row["Korte omschrijving"])
    return word in split_keywords(row[SOURCE[block]])

# test: maximaal k, aflopend, alleen > 0, en elk genoemd woord komt echt in de titel voor
for person in readers:
    p = profile_for(person)
    for code in recommend(person, n=3):
        result = explain(p, code, k=3)
        assert 1 <= len(result) <= 3, (person["name"], code, result)
        values = [v for _, v in result]
        assert values == sorted(values, reverse=True) and all(v > 0 for v in values)
        assert all(occurs(name, code) for name, _ in result), (person["name"], code, result)
assert len(explain(profile_for(daan), recommend(daan, n=1)[0], k=1)) == 1
print("explain klopt")

top1 = recommend(thomas, n=1)[0]
print(catalog.loc[top1, "Titel"], "->", explain(profile_for(thomas), top1, k=3))

## 5 · Welk blok droeg bij?

Soms is het handiger om te weten **uit welk blok** de match komt dan welke losse woorden: kwam deze titel in de lijst door zijn omschrijving, door zijn genre, of door zijn trefwoorden? Dat is dezelfde bijdragenrij, nu per blok opgeteld en gedeeld door de totale cosine. De aandelen tellen dus op tot 1.

Om de som per blok te maken heb je de **grenzen van de blokken** nodig in `X`. Haal die uit de breedtes van de vectorizers (de blokken zijn in die volgorde achter elkaar geplakt), niet door namen te tellen of op een beginletter te filteren: dan knip je op de verkeerde plek.

In [ ]:
WIDTHS = [len(VECS[b].get_feature_names_out()) for b in BLOCKS]
EDGES = np.cumsum([0] + WIDTHS)                       # blok i = kolommen EDGES[i] t/m EDGES[i+1]
print(dict(zip(BLOCKS, WIDTHS)), "grenzen:", EDGES.tolist())
assert EDGES[-1] == X.shape[1]

> **Opdracht 4 · explain_by_block.** Schrijf `explain_by_block(profile, code)`: een dict met per blok het aandeel in de cosine (tekst, genre, trefwoorden), die samen 1 zijn. Is de cosine 0 (geen enkel woord gemeen), geef dan voor elk blok 0,0 terug.
>
> **Hier verwacht ik een antwoord op:** `explain_by_block` die de test doorstaat · voor twee testlezers het blokaandeel van hun nummer 1, en wat dat zegt over hoe de lezer zijn wens invulde (alleen `free_text`, of ook tags en genres)

In [ ]:
def explain_by_block(profile, code):
    """The share of each block (tekst, genre, trefwoorden) in the cosine between profile and title."""
    # - Bereken de bijdragen met contributions(profile, code)
    # - Tel per blok de bijdragen op; gebruik EDGES voor de grenzen van het blok
    # - Deel elke som door de totale som, zodat de aandelen samen 1 zijn
    # - Is de totale som 0? Geef dan voor elk blok 0.0 terug.
    raise NotImplementedError

In [ ]:
for person in readers:
    p = profile_for(person)
    for code in recommend(person, n=3):
        share = explain_by_block(p, code)
        assert list(share) == BLOCKS
        assert abs(sum(share.values()) - 1) < 1e-9, (person["name"], code, share)
assert explain_by_block(profile_for({"free_text": "xyzzy", "level": "2F"}), CODES[0]) == {b: 0.0 for b in BLOCKS}
print("explain_by_block klopt")

pd.DataFrame([{"lezer": person["name"], **{b: round(v, 2) for b, v in explain_by_block(profile_for(person), recommend(person, n=1)[0]).items()}}
              for person in readers])

## 6 · Uitleg voor de zeven testlezers

Hieronder staat voor elke testlezer de nummer 1, met de drie woorden en het blokaandeel. Lees ze zoals de lezer zelf zou doen: zou ik dit als reden herkennen? De zeven lezers zijn niet toevallig gekozen, en bij een paar valt iets op:

- **Thomas** schrijft "Tweede Wereldoorlog" in zijn eigen woorden, maar zijn trefwoorden zijn "WOII" en "oorlog". Wat voor soort woorden zie je terug in de uitleg?
- **Lotte** schrijft over "rouw"; de catalogus zegt vaker "verlies". Kan een uitleg een woord noemen dat zij niet gebruikte?
- **Milan** wil sport én AI. Komt de uitleg uit één van de twee, of uit allebei?

In [ ]:
for person in readers:
    p = profile_for(person)
    code_1 = recommend(person, n=1)[0]
    share = {b: round(v, 2) for b, v in explain_by_block(p, code_1).items()}
    print(f"{person['name']:8s} -> {catalog.loc[code_1, 'Titel']}")
    print(f"           woorden: {explain(p, code_1, k=3)}")
    print(f"           blokken: {share}")

> **Opdracht 5 · een opvallende uitleg.** Draai de cel hierboven voor de zeven testlezers en voeg je eigen testlezer toe uit het vorige notebook. Zoek **één** uitleg die je verrast of waar iets niet klopt: een woord dat nergens op slaat, een uitleg uit het verkeerde blok, of een goede aanbeveling met een zwakke reden. Zoek in de omschrijving van die titel (`catalog.loc[code, "Korte omschrijving"]`) uit hoe het kan.
>
> **Hier verwacht ik een antwoord op:** de lezer, de titel en de uitleg die opviel · wat de oorzaak is (stopwoord, ontbrekend trefwoord, een woord dat in veel omschrijvingen staat, ...) · wat je eraan zou veranderen in je recommender of in je catalogus

## 7 · Van getallen naar een zin voor de lezer

`explain` geeft `[("tekst:oorlog", 0.31), ("trefwoorden:woii", 0.18), ...]`. Dat is een uitleg voor jou, niet voor een lezer. Op het scherm komt een **zin**. Je hebt daarbij keuzes te maken:

| Keuze | Opties |
|---|---|
| **Wat toon je?** | de woorden, de blokken ("past bij je genre"), de dichtstbijzijnde titel die de lezer koos, de score, of een combinatie |
| **Hoeveel?** | één reden is duidelijk, vijf is een lijst die niemand leest; kies `k` |
| **Wat laat je weg?** | woorden zonder betekenis voor de lezer, bijdragen die erg klein zijn, het genoemde blok als de lezer dat veld niet invulde |
| **Welke toon?** | "omdat jij ... noemde" (alleen waar het ook zo is) of "deze titel gaat over ..." (neutraler) |

Een uitleg moet **waar** blijven: schrijf nooit "omdat je spanning leuk vindt" als de match uit het woord "geheim" in de omschrijving komt.

> **Opdracht 6 · explain_text.** Schrijf `explain_text(person, code, k=3)`: een zin voor de lezer op basis van `explain`. Maak een eigen keuze uit de tabel hierboven. De test controleert alleen dat de zin een tekst is en het belangrijkste woord noemt; de rest is jouw keuze.
>
> **Hier verwacht ik een antwoord op:** `explain_text` die de test doorstaat · de zin voor de nummer 1 van drie testlezers · keuzelog-rij: welke uitleg op het scherm, welke alternatieven je overwoog, en de reden

In [ ]:
def explain_text(person, code, k=3):
    """One sentence for the reader on why this title is in the list."""
    # - Bouw het profiel van de lezer: profile_for(person)
    # - Haal de woorden op met explain(profile, code, k)
    # - Splits elke naam in blok en woord: name.split(":", 1)
    # - Maak er een zin van; laat het blok weg of noem het, naar jouw keuze
    # - Is er geen enkel woord (lijst leeg)? Geef een zin terug die dat eerlijk zegt.
    raise NotImplementedError

In [ ]:
for person in readers:
    code_1 = recommend(person, n=1)[0]
    sentence = explain_text(person, code_1)
    assert isinstance(sentence, str) and len(sentence) > 0
    top_word = explain(profile_for(person), code_1, k=1)[0][0].split(":", 1)[1]
    assert top_word in sentence, (person["name"], top_word, sentence)
print("explain_text klopt")

for person in [thomas, lotte, milan]:
    code_1 = recommend(person, n=1)[0]
    print(f"{person['name']:8s} {catalog.loc[code_1, 'Titel']}: {explain_text(person, code_1)}")

## 8 · Uitleg als hulpmiddel: waarom ontbreekt deze titel?

Een uitleg is ook een **debughulpmiddel**. In de evaluatie zag je lezers met een lage recall: titels die wel in `relevant` staan maar niet in de top 5 komen. `explain` kan antwoord geven op "waarom niet?": kijk welke woorden de gemiste titel wél deelt met het profiel, en welke woorden uit het profiel ontbreken in de titel.

Hieronder voor Thomas. `top_profile_words` geeft de zwaarste kolommen van het profiel, en `missing_words` welke daarvan de titel niet heeft.

In [ ]:
def top_profile_words(profile, k=8):
    """The k heaviest columns of a profile, with their weight."""
    row = profile.toarray().ravel()
    return [(NAMES[j], round(float(row[j]), 3)) for j in np.argsort(-row)[:k] if row[j] > 0]

def missing_words(profile, code, k=8):
    """Heavy profile columns that the title does not have: why does this title score lower?"""
    row = X[POS[code]].toarray().ravel()
    return [(name, w) for name, w in top_profile_words(profile, 3 * k) if row[NAMES.index(name)] == 0][:k]

person = thomas
p = profile_for(person)
top5 = recommend(person, n=5)
missed = [c for c in person["relevant"] if c not in top5]
print(f"{person['name']}: {len(missed)} van {len(person['relevant'])} relevante titels niet in de top 5\n")

# de gemiste titel met de hoogste cosine: dichtbij, maar net niet
sim = dict(zip(CODES, cosine_similarity(p, X).ravel()))
code_missed = max(missed, key=sim.get)
print("gemist: ", catalog.loc[code_missed, "Titel"], "| cosine", round(sim[code_missed], 3))
print("  wel gemeen:    ", explain(p, code_missed, k=5))
print("  profiel mist:  ", missing_words(p, code_missed, k=5))
print("nummer 5:", catalog.loc[top5[-1], "Titel"], "| cosine", round(sim[top5[-1]], 3))
print("  wel gemeen:    ", explain(p, top5[-1], k=5))

> **Reflectie.** Kijk naar de gemiste titel hierboven en naar zijn omschrijving en trefwoorden (`catalog.loc[code_missed]`). Welke woorden zou je moeten toevoegen of aanpassen zodat hij in de top 5 komt? Verandert dat de **recommender** (gewichten, stopwoorden, een synoniemenlijst) of de **catalogus** (trefwoorden aanvullen)? Noteer je antwoord in het keuzelog bij je recall-cijfers uit de evaluatie.

---
## Opdracht

**Doel.** Jullie systeem laat bij elke aanbeveling zien waarom, in woorden die waar zijn en die de lezer begrijpt.

**Moet** (in `recommender.py`)
- [ ] `contributions(profile, code)`: de bijdrage per kolom aan de cosine, optellend tot de cosine.
- [ ] `feature_names(vecs)`: de namen van alle kolommen van `X`, in de volgorde van jullie blokken, met blokprefix. Een assert op `len(names) == X.shape[1]`.
- [ ] `explain(profile, code, k=3)`: de `k` woorden of categorieën met de grootste bijdrage (> 0).
- [ ] `explain_by_block(profile, code)`: het aandeel van elk blok in de cosine, samen 1.
- [ ] `explain_text(person, code)`: een zin voor de lezer, met jouw gekozen vorm in het keuzelog.

**Kan**
- [ ] De uitleg "omdat je *titel X* koos": zoek bij een aanbeveling de titel uit `liked_titles` met de hoogste cosine tot de aanbeveling, en noem die.
- [ ] Een uitleg voor de koude start (geen `liked_titles`): wat zeg je tegen een lezer van wie je niets weet?

*Klaar als.* Voor elke testlezer toont jullie app een nummer 1 met een uitleg, de uitleg klopt met de omschrijving van die titel (`occurs`), en in het keuzelog staat welke vorm je koos en waarom.

### Startpunt voor `recommender.py`

Jullie kolommen en blokken heten anders; haal de namen uit jullie eigen `build_vectors`. Het belangrijkste: `feature_names` moet **dezelfde blokken in dezelfde volgorde** nemen als `build_vectors`, en `EDGES` komt uit de breedtes van jullie blokken.

```python
def contributions(profile, code):
    # profiel × titelrij, gedeeld door beide lengtes; optellend tot de cosine
    raise NotImplementedError

def feature_names(vecs):
    # blokprefix + woord, in dezelfde volgorde als de blokken in X
    raise NotImplementedError

def explain(profile, code, k=3):
    # de k grootste bijdragen (> 0) als (naam, waarde)
    raise NotImplementedError

def explain_by_block(profile, code):
    # per blok de som van de bijdragen, gedeeld door het totaal
    raise NotImplementedError

def explain_text(person, code, k=3):
    # één zin voor de lezer, op basis van explain
    raise NotImplementedError
```